# GOT-OCR2.0 on Colab (KAE)

Запускает **GOT-OCR2.0** (stepfun-ai/GOT-OCR2_0) через transformers на Colab-GPU:
документ/схема → текст и структурированный вывод (Markdown/LaTeX для таблиц и
формул). Ячейки 1–4 — интерактивный тест на загруженной картинке; ячейки 5–6 —
HTTP-сервис + туннель, чтобы KAE слал регионы на распознавание.

**Перед запуском:** Runtime → Change runtime type → **T4 GPU** (не TPU!).

In [ ]:
# 1. GPU
!nvidia-smi -L || print('Поставь T4 GPU: Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# 2. Зависимости (версии, с которыми GOT-OCR2.0 стабилен)
!pip -q install transformers==4.37.2 tiktoken==0.6.0 verovio accelerate torchvision

In [ ]:
# 3. Загрузка модели (~1.4GB, на T4)
import torch
from transformers import AutoModel, AutoTokenizer
tok = AutoTokenizer.from_pretrained('stepfun-ai/GOT-OCR2_0', trust_remote_code=True)
model = AutoModel.from_pretrained(
    'stepfun-ai/GOT-OCR2_0', trust_remote_code=True, low_cpu_mem_usage=True,
    device_map='cuda', use_safetensors=True, pad_token_id=tok.eos_token_id
).eval().cuda()
print('GOT-OCR2.0 готов')

In [ ]:
# 4. Тест на картинке: загрузи присланный final50.jpg (или любую схему/таблицу)
from google.colab import files
up = files.upload()
p = list(up.keys())[0]
print('=== PLAIN OCR (весь текст) ===')
print(model.chat(tok, p, ocr_type='ocr'))
print('\n=== FORMATTED (markdown/latex, структура) ===')
print(model.chat(tok, p, ocr_type='format'))
# Для больших/плотных страниц лучше режим с нарезкой:
# print(model.chat_crop(tok, p, ocr_type='format'))

In [ ]:
# 5. HTTP-сервис: POST /ocr {image_b64, ocr_type} -> {text}. Для интеграции с KAE.
from flask import Flask, request, jsonify
import base64, tempfile, threading, os
app = Flask(__name__)

@app.post('/ocr')
def ocr():
    d = request.get_json(force=True)
    raw = base64.b64decode(d['image_b64'])
    fd, path = tempfile.mkstemp(suffix='.png'); os.write(fd, raw); os.close(fd)
    try:
        text = model.chat(tok, path, ocr_type=d.get('ocr_type', 'format'))
    finally:
        os.remove(path)
    return jsonify({'text': text})

threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5005), daemon=True).start()
print('GOT-OCR HTTP-сервис на :5005')

In [ ]:
# 6. Туннель наружу — публичный URL сервиса GOT-OCR для KAE
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import subprocess, re, itertools
proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5005'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in itertools.islice(proc.stdout, 200):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\n' + '=' * 60)
print('GOT-OCR URL:', url)
print('POST', (url or '<url>') + '/ocr  body={"image_b64":"...","ocr_type":"format"}')
print('=' * 60)
import time
while True: time.sleep(300)  # держать сессию живой